# Qwen3.8-27B · vLLM · 300K context · OpenAI-compatible API

**執行階段**:`G4 GPU`(首選,Blackwell 96GB,跑 NVFP4)或 `A100 GPU` + 開啟「大量 RAM」(80GB,跑 FP8)。L4 / T4 / A100 40GB 跑不了 300K。

由上往下依序執行。最後一格會關掉執行階段,停止扣運算單元。

## 1. 設定

In [ ]:
MAX_LEN = 307_200          # 300K tokens(原生 262,144,超過部分靠 vLLM 延伸)
SERVED_NAME = "qwen3.8-27b"
PORT = 8000
MAX_SEQS = 2               # 同時處理的請求數;長 context 下保守一點

import secrets
API_KEY = secrets.token_urlsafe(24)   # 給 omp 用的金鑰,執行後會印出
print("API_KEY =", API_KEY)

## 2. 檢查 GPU、選 checkpoint

In [ ]:
import subprocess
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"]
).decode().strip()
name, mem_mib = [x.strip() for x in gpu.split(",")]
mem_gb = int(mem_mib) / 1024
print(f"GPU: {name}  {mem_gb:.1f} GB")

if "Blackwell" in name or "RTX PRO 6000" in name:
    MODEL = "Inferact/Qwen3.8-27B-NVFP4"     # vLLM 官方 recipe:約 24.6 GiB
    EXTRA = ["--kv-cache-dtype", "fp8"]
elif "A100" in name and mem_gb > 70:
    MODEL = "Qwen/Qwen3.8-27B-FP8"           # Ampere 無原生 FP8,vLLM 走 weight-only 路徑
    EXTRA = []
else:
    raise SystemExit("這張卡撐不住 300K:請換 G4,或 A100 並開啟「大量 RAM」(80GB)")
print("MODEL:", MODEL)

## 3. 安裝 vLLM(約 3–5 分鐘)

In [ ]:
!pip install -q -U uv
!uv pip install --system -q -U vllm hf_transfer
import os; os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
!python -c "import vllm; print('vLLM', vllm.__version__)"

## 4. 啟動 vLLM server(背景執行,首次會下載權重)

In [ ]:
import json, time, urllib.request

cmd = [
    "vllm", "serve", MODEL,
    "--served-model-name", SERVED_NAME,
    "--port", str(PORT),
    "--api-key", API_KEY,
    "--max-model-len", str(MAX_LEN),
    "--hf-overrides", json.dumps({"text_config": {"max_position_embeddings": MAX_LEN}}),
    "--max-num-seqs", str(MAX_SEQS),
    "--gpu-memory-utilization", "0.92",
    "--enable-prefix-caching",
    "--reasoning-parser", "qwen3",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder",
    *EXTRA,
]
print(" ".join(cmd))
log = open("/content/vllm.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

def ready():
    try:
        req = urllib.request.Request(f"http://localhost:{PORT}/v1/models",
                                     headers={"Authorization": f"Bearer {API_KEY}"})
        return urllib.request.urlopen(req, timeout=3).status == 200
    except Exception:
        return False

t0 = time.time()
while not ready():
    if server.poll() is not None:
        raise SystemExit("vLLM 掛了,看下一格的 log")
    time.sleep(10)
    print(f"\r等待中… {int(time.time()-t0)}s", end="")
print(f"\n✅ server ready ({int(time.time()-t0)}s)")

In [ ]:
# 出錯或想看 KV cache 容量時執行(找 'Maximum concurrency' / 'KV cache' 那幾行)
!grep -iE "kv cache|maximum concurrency|error" /content/vllm.log | tail -n 20

## 5. 本機測試

In [ ]:
!pip install -q openai
from openai import OpenAI
client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key=API_KEY)
r = client.chat.completions.create(
    model=SERVED_NAME,
    messages=[{"role": "user", "content": "用一句話介紹你自己"}],
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(r.choices[0].message.content)

## 6. 對外開放(Cloudflare quick tunnel,給 omp 連)

In [ ]:
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
import re
tlog = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
                          stdout=tlog, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m: url = m.group(0); break
    time.sleep(2)
print("BASE_URL =", f"{url}/v1")
print("API_KEY  =", API_KEY)
print("MODEL    =", SERVED_NAME)

omp 端填 OpenAI-compatible provider:上面三個值。tunnel 網址每次重開都會變。

**注意**:Colab Pro 沒有背景執行,瀏覽器分頁要保持開著;只有外部 API 流量不算 Colab 互動,閒置太久仍可能被斷線。

## 7. 收工(停止扣運算單元)

In [ ]:
for p in ("tunnel", "server"):
    if p in globals(): globals()[p].terminate()
from google.colab import runtime
runtime.unassign()